In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import geopandas as gpd
import planetary_computer as pc
import pystac_client
import pandas as pd

CONFIG_DIR = project_root / "config"
RAW_DIR = project_root / "data" / "raw" / "sentinel-2"

RAW_DIR.mkdir(parents=True, exist_ok=True)

STUDY_AREA_PATH = CONFIG_DIR / "study_area.geojson"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SENTINEL2_COLLECTION = "sentinel-2-l2a"

print("Project root:", project_root)
print("Study area:", STUDY_AREA_PATH)
print("Raw data folder:", RAW_DIR)

In [2]:
import geopandas as gpd

study_area = gpd.read_file("../config/study_area.geojson")

print(study_area)
print("\nCRS:", study_area.crs)
print("\nBounds:", study_area.total_bounds)


In [3]:
import planetary_computer as pc
import pystac_client

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SENTINEL2_COLLECTION = "sentinel-2-l2a"

catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=pc.sign_inplace,
)

print("Connected to Planetary Computer STAC catalog")
print("Collection:", SENTINEL2_COLLECTION)

In [4]:
# Convert the study area to WGS84 and calculate its bounding box
if study_area.crs is None or study_area.crs.to_epsg() != 4326:
    study_area = study_area.to_crs(epsg=4326)

study_area_bounds = study_area.total_bounds

# total_bounds returns [minx, miny, maxx, maxy]
bbox = [
    study_area_bounds[0],
    study_area_bounds[1],
    study_area_bounds[2],
    study_area_bounds[3],
]

print("Study-area bounding box:")
print(bbox)

In [5]:
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
MAX_CLOUD_COVER = 20

search = catalog.search(
    collections=[SENTINEL2_COLLECTION],
    bbox=bbox,
    datetime=f"{START_DATE}/{END_DATE}",
    query={
        "eo:cloud_cover": {"lt": MAX_CLOUD_COVER}
    },
)

items = list(search.items())

print(f"Found {len(items)} Sentinel-2 L2A scenes")

In [6]:
scene_records = []

for item in items:
    scene_records.append(
        {
            "scene_id": item.id,
            "datetime": item.datetime,
            "cloud_cover": item.properties.get("eo:cloud_cover"),
            "mgrs_tile": item.properties.get("s2:mgrs_tile"),
            "platform": item.properties.get("platform"),
            "collection": item.collection_id,
        }
    )

scenes = pd.DataFrame(scene_records)
scenes = scenes.sort_values("datetime")

scenes

In [7]:
first_item = items[0]

print("Scene ID:", first_item.id)
print("Date:", first_item.datetime)
print("\nAvailable assets:")

for asset_key in sorted(first_item.assets.keys()):
    print(asset_key)

In [8]:
metadata_path = RAW_DIR / "sentinel2_scene_catalog_2024.csv"
scenes.to_csv(metadata_path, index=False)

print(f"Saved {len(scenes)} scene records to:")
print(metadata_path)